# Fake News Detection — EDA & Model Training

This notebook walks through data exploration, preprocessing, model training, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import sys, os
sys.path.append('..')

from utils.text_cleaner import clean_text

df = pd.read_csv('../dataset/news.csv')
print('Shape:', df.shape)
df.head()

## 1. Label Distribution

In [ ]:
plt.figure(figsize=(6,4))
df['label'].value_counts().plot(kind='bar', color=['#00e676','#ff3b6b'], edgecolor='black')
plt.xticks([0,1], ['REAL','FAKE'], rotation=0)
plt.title('Class Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 2. Word Count Distribution

In [ ]:
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))
df.groupby('label')['word_count'].hist(bins=50, alpha=0.6, figsize=(8,4))
plt.xlabel('Word Count')
plt.title('Word Count Distribution by Label')
plt.legend(['REAL','FAKE'])
plt.show()

## 3. Word Clouds

In [ ]:
def generate_wordcloud(text, title, color):
    wc = WordCloud(width=600, height=300, background_color='white', colormap=color, max_words=100).generate(text)
    plt.figure(figsize=(8,4))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=14)
    plt.show()

fake_text = ' '.join(df[df['label']==1]['text'].dropna().sample(min(500,len(df[df['label']==1]))))
real_text = ' '.join(df[df['label']==0]['text'].dropna().sample(min(500,len(df[df['label']==0]))))

generate_wordcloud(fake_text, 'Fake News Word Cloud', 'Reds')
generate_wordcloud(real_text, 'Real News Word Cloud', 'Greens')

## 4. Train Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

df['clean'] = df['text'].apply(clean_text)
X_train, X_test, y_train, y_test = train_test_split(df['clean'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X_tr = vectorizer.fit_transform(X_train)
X_te = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, C=1.0)
model.fit(X_tr, y_train)

y_pred = model.predict(X_te)
print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print(classification_report(y_test, y_pred, target_names=['REAL','FAKE']))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['REAL','FAKE'], yticklabels=['REAL','FAKE'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 6. Save Model

In [ ]:
os.makedirs('../models', exist_ok=True)
with open('../models/fake_news_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
print('Model saved!')